# 08. Error analysis on the cold splits: which proteins and families, and how sure are we?

The overall CI and MSE hide where the models work and where they fail. This notebook re-scores the finished full-data models with the `--predictions-only` flag (nothing is retrained and no existing result file is touched), then analyzes the saved per-row predictions:

- sanity check: MSE and CI recomputed from the saved predictions against the numbers already reported;
- per held-out protein: MSE and Spearman correlation across its 68 drugs (can the model rank the drugs for a new kinase?);
- per cluster (family on `cold_family`): the same, aggregated;
- paired cluster bootstrap: the difference between two arms with a 95% interval that accounts for which 88 proteins happened to be held out. This is a different and more honest uncertainty than the seed spread: seeds only vary initialization, not the choice of held-out proteins.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Config

In [ ]:
import os
from glob import glob

def find_folder(name):
    path = f"/content/drive/MyDrive/{name}"
    if not os.path.isdir(path):
        shared = glob(f"/content/drive/.shortcut-targets-by-id/*/{name}")
        if shared:
            path = shared[0]
    assert os.path.isdir(path), f"Folder not found: {path}"
    return path

DRIVE_ROOT = find_folder("Diplomska_hybrid")
COLD_ROOT = find_folder("Diplomska_cold")
PDB_SET = "kinase_domain"
SPLITS = ["cold_protein", "cold_family"]
SEEDS = [42, 43, 44]

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("working folder:", DRIVE_ROOT)
print("cold folder (read only):", COLD_ROOT)

working folder: /content/drive/MyDrive/Diplomska_hybrid
cold folder (read only): /content/drive/MyDrive/Diplomska_cold


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages

In [ ]:
!pip install -q torch_geometric biopython esm

## 5. Go to the folder

In [ ]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/MyDrive/Diplomska_hybrid
/content/drive/MyDrive/Diplomska_hybrid


## 6. Check the uploaded code

In [ ]:
import argparse, json
import numpy as np
import pandas as pd

required = ["training_5_folds.py", "testing.py", "compute_protein_embeddings.py",
            "src/common/predictions.py", "src/common/run_tag.py", "src/common/paths.py",
            "src/hybrid/hybrid_net.py", "src/structure/gnn_net.py", "src/sequence/plm_net.py"]
print("Missing code files:", [f for f in required if not os.path.exists(f)] or "none")
with open("testing.py") as f:
    text = f.read()
print("--save-predictions in testing.py:", "--save-predictions" in text)
print("--predictions-only in testing.py:", "--predictions-only" in text)
assert "--predictions-only" in text, "Upload the updated testing.py"

from src.common.run_tag import run_tag
from src.common.paths import paths_for
PATHS_BY_SPLIT = {s: paths_for("davis", PDB_SET, split=s) for s in SPLITS}
for s, p in PATHS_BY_SPLIT.items():
    print(s, "| root ok:", p.root == DRIVE_ROOT, "| folds file:", os.path.exists(p.train_folds), "| test file:", os.path.exists(p.test_fold))

Missing code files: none
--save-predictions in testing.py: True
--predictions-only in testing.py: True
cold_protein | root ok: True | folds file: True | test file: True
cold_family | root ok: True | folds file: True | test file: True


## 7. Copy models and embeddings from Diplomska_cold

In [ ]:
import shutil, time

def copy_file(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return False
    shutil.copyfile(src, dst)
    return True

# models
src_models = os.path.join(COLD_ROOT, "models_davis")
dst_models = PATHS_BY_SPLIT[SPLITS[0]].models
os.makedirs(dst_models, exist_ok=True)
t0 = time.time(); copied = skipped = 0
for name in sorted(os.listdir(src_models)):
    if name.endswith("_full.model") and ("pdb2_cprot" in name or "pdb2_cfam" in name):
        if copy_file(os.path.join(src_models, name), os.path.join(dst_models, name)):
            copied += 1
        else:
            skipped += 1
print(f"models: copied {copied}, skipped {skipped}, {time.time() - t0:.0f} s")

# embeddings of the other arms
emb_src = os.path.join(COLD_ROOT, "data", "davis", "protein_embeddings_kinase_domain")
emb_dst = PATHS_BY_SPLIT[SPLITS[0]].protein_embeddings
for m in ["esmc_600m", "prot_t5_xl_uniref50", "random_1024"]:
    s_dir, d_dir = os.path.join(emb_src, m), os.path.join(emb_dst, m)
    if not os.path.isdir(s_dir):
        print(f"{m}: not found in cold folder (random_1024 is rebuilt below if missing)")
        continue
    os.makedirs(d_dir, exist_ok=True)
    t0 = time.time(); c = 0
    for name in os.listdir(s_dir):
        if copy_file(os.path.join(s_dir, name), os.path.join(d_dir, name)):
            c += 1
    print(f"{m}: copied {c} files, {len(glob(os.path.join(d_dir, '*.pooled.pt')))} pooled vectors, {time.time() - t0:.0f} s")

models: copied 29, skipped 1, 18 s
esmc_600m: copied 429 files, 428 pooled vectors, 175 s
prot_t5_xl_uniref50: copied 429 files, 428 pooled vectors, 156 s
random_1024: copied 429 files, 428 pooled vectors, 183 s


In [ ]:
import subprocess
if len(glob(os.path.join(emb_dst, "random_1024", "*.pooled.pt"))) < 428:
    subprocess.run(["python", "-u", "compute_protein_embeddings.py", "--model", "random_1024", "--pdb-set", PDB_SET], check=True)
    print("random_1024 vectors:", len(glob(os.path.join(emb_dst, "random_1024", "*.pooled.pt"))), "(expected 428)")
else:
    print("random_1024 vectors already present")

random_1024 vectors already present


## 8. Helpers

In [ ]:
import re, statistics, subprocess

PART = {"cold_family": "cfam", "cold_protein": "cprot"}
ARMS = {
    "random_1024": ("plm", "PLMNet", "random_1024"),
    "esmc_300m": ("plm", "PLMNet", "esmc_300m"),
    "esmc_600m": ("plm", "PLMNet", "esmc_600m"),
    "prot_t5_xl_uniref50": ("plm", "PLMNet", "prot_t5_xl_uniref50"),
    "structure": ("structure", "GNNNet", None),
    "hybrid": ("hybrid", "HybridNet", "esmc_300m"),
    "hybrid_noedges": ("hybrid_noedges", "HybridNet", "esmc_300m"),
}

def arm_tag(arm, split, seed):
    repr_, _, plm = ARMS[arm]
    t = run_tag(repr_, plm, seed, False, PDB_SET, split=split)
    assert PART[split] in t, t
    return t

def flags(arm, split, seed):
    repr_, _, plm = ARMS[arm]
    out = ["--protein-repr", repr_, "--pdb-set", PDB_SET, "--split", split]
    if plm:
        out += ["--plm-model", plm]
    if seed != 42:
        out += ["--seed", str(seed)]
    return out

def model_file(arm, split, seed):
    return os.path.join(PATHS_BY_SPLIT[split].models, f"model_{ARMS[arm][1]}_davis_{arm_tag(arm, split, seed)}_full.model")

def pred_file(arm, split, seed):
    return os.path.join(PATHS_BY_SPLIT[split].results, f"predictions_{ARMS[arm][1]}_davis_{arm_tag(arm, split, seed)}_full.csv")

def reported_file(arm, split, seed):
    name = f"result_{ARMS[arm][1]}_davis_{arm_tag(arm, split, seed)}_full.txt"
    for folder in [PATHS_BY_SPLIT[split].results, os.path.join(COLD_ROOT, "results_davis"), os.path.join(DRIVE_ROOT, "reference_cold")]:
        if os.path.exists(os.path.join(folder, name)):
            return os.path.join(folder, name)
    return None

def read_mean(path, metric):
    if not path or not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if line.startswith(metric + " ") and "mean:" in line:
                return float(line.split("mean:")[1].split()[0])
    return None

def concordance(y, p, chunk=500):
    y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
    num = 0.0; den = 0
    for i in range(0, len(y), chunk):
        dy = y[i:i + chunk, None] - y[None, :]
        dp = p[i:i + chunk, None] - p[None, :]
        pairs = dy > 0
        num += float((pairs & (dp > 0)).sum()) + 0.5 * float((pairs & (dp == 0)).sum())
        den += int(pairs.sum())
    return num / den

# Re-score the finished full-data models
`--predictions-only` writes only `predictions_*_full.csv` and no result file or plot. About a minute per model. Existing CSVs and missing models are skipped.

In [ ]:
missing, done, failed = [], 0, []
for split in SPLITS:
    for arm in ARMS:
        for seed in SEEDS:
            if not os.path.exists(model_file(arm, split, seed)):
                missing.append((split, arm, seed)); continue
            if os.path.exists(pred_file(arm, split, seed)):
                done += 1; continue
            cmd = ["python", "-u", "testing.py"] + flags(arm, split, seed) + ["--full", "--predictions-only"]
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0 or not os.path.exists(pred_file(arm, split, seed)):
                failed.append((split, arm, seed)); print("FAILED:", " ".join(cmd)); print(r.stdout[-1500:]); print(r.stderr[-1500:])
            else:
                done += 1; print("scored:", split, arm, seed, flush=True)
print(f"prediction files available: {done} | missing models (skipped): {len(missing)} | failed: {len(failed)}")
for m in missing:
    print("  no model:", m)

scored: cold_protein random_1024 42
scored: cold_protein random_1024 43
scored: cold_protein random_1024 44
scored: cold_protein esmc_300m 42
scored: cold_protein esmc_300m 43
scored: cold_protein esmc_300m 44
scored: cold_protein esmc_600m 42
scored: cold_protein esmc_600m 43
scored: cold_protein esmc_600m 44
scored: cold_protein prot_t5_xl_uniref50 42
scored: cold_protein prot_t5_xl_uniref50 43
scored: cold_protein prot_t5_xl_uniref50 44
scored: cold_protein structure 42
scored: cold_protein structure 43
scored: cold_protein structure 44
scored: cold_protein hybrid 42
scored: cold_protein hybrid 43
scored: cold_protein hybrid 44
scored: cold_protein hybrid_noedges 42
scored: cold_protein hybrid_noedges 43
scored: cold_protein hybrid_noedges 44
scored: cold_family random_1024 42
scored: cold_family random_1024 43
scored: cold_family random_1024 44
scored: cold_family esmc_300m 42
scored: cold_family esmc_300m 43
scored: cold_family esmc_300m 44
scored: cold_family esmc_600m 42
scored:

# Load predictions and check

In [ ]:
PRED = {}   # (split, arm, seed) -> dataframe
for split in SPLITS:
    for arm in ARMS:
        for seed in SEEDS:
            f = pred_file(arm, split, seed)
            if os.path.exists(f):
                PRED[(split, arm, seed)] = pd.read_csv(f)

print(f"{'split':13} {'arm':20} {'seed':>4} {'rows':>5} {'MSE csv':>8} {'MSE reported':>12} {'CI csv':>7} {'CI reported':>11}")
for (split, arm, seed), d in sorted(PRED.items()):
    mse = float(np.mean((d.label - d.prediction) ** 2)); ci = concordance(d.label, d.prediction)
    rf = reported_file(arm, split, seed)
    rm, rc = read_mean(rf, "mse"), read_mean(rf, "ci")
    flag = "" if (rm is None or (abs(mse - rm) < 2e-3 and abs(ci - rc) < 2e-3)) else "  <-- CHECK"
    print(f"{split:13} {arm:20} {seed:>4} {len(d):>5} {mse:8.4f} {rm if rm is not None else float('nan'):12.4f} {ci:7.4f} {rc if rc is not None else float('nan'):11.4f}{flag}")

split         arm                  seed  rows  MSE csv MSE reported  CI csv CI reported
cold_family   esmc_300m              42  5984   0.4358       0.4358  0.7716      0.7716
cold_family   esmc_300m              43  5984   0.3986       0.3986  0.7890      0.7890
cold_family   esmc_300m              44  5984   0.4885       0.4885  0.7815      0.7815
cold_family   esmc_600m              42  5984   0.4118       0.4118  0.7819      0.7819
cold_family   esmc_600m              43  5984   0.4301       0.4301  0.7741      0.7741
cold_family   esmc_600m              44  5984   0.5786       0.5786  0.7382      0.7382
cold_family   prot_t5_xl_uniref50    42  5984   0.4665       0.4665  0.7385      0.7385
cold_family   prot_t5_xl_uniref50    43  5984   0.4671       0.4671  0.7483      0.7483
cold_family   prot_t5_xl_uniref50    44  5984   1.0817       1.0817  0.7101      0.7101
cold_family   random_1024            42  5984   1.1696       1.1696  0.6669      0.6669
cold_family   random_1024       

# Seed-averaged predictions and the no-protein floor

In [ ]:
df_all = pd.read_csv(PATHS_BY_SPLIT[SPLITS[0]].csv)
AVG = {}   # split -> {arm: dataframe(row_index, protein, ligand, label, prediction)}
for split in SPLITS:
    P = PATHS_BY_SPLIT[split]
    with open(P.train_folds) as f: train_idx = [int(i) for fold in json.load(f) for i in fold]
    with open(P.test_fold) as f: test_idx = [int(i) for i in json.load(f)]
    train, test = df_all.iloc[train_idx], df_all.iloc[test_idx].copy()
    dm = train.groupby("ligand")["label"].mean()
    test["prediction"] = test["ligand"].map(dm).fillna(train["label"].mean()).to_numpy()
    test["row_index"] = test.index
    AVG[split] = {"drug_mean": test[["row_index", "protein", "ligand", "label", "prediction"]].reset_index(drop=True)}
    for arm in ARMS:
        parts = [PRED[(split, arm, s)] for s in SEEDS if (split, arm, s) in PRED]
        if not parts:
            continue
        base = parts[0][["row_index", "protein", "ligand", "label"]].copy()
        for p in parts[1:]:
            assert (p["row_index"].to_numpy() == base["row_index"].to_numpy()).all(), "row order differs between seeds"
        base["prediction"] = np.mean([p["prediction"].to_numpy() for p in parts], axis=0)
        AVG[split][arm] = base
        AVG[split][arm].attrs["n_seeds"] = len(parts)
    print(split, {a: AVG[split][a].attrs.get("n_seeds", "-") for a in AVG[split]})

cold_protein {'drug_mean': '-', 'random_1024': 3, 'esmc_300m': 3, 'esmc_600m': 3, 'prot_t5_xl_uniref50': 3, 'structure': 3, 'hybrid': 3, 'hybrid_noedges': 3}
cold_family {'drug_mean': '-', 'random_1024': 3, 'esmc_300m': 3, 'esmc_600m': 3, 'prot_t5_xl_uniref50': 3, 'structure': 3}


# Per-protein metrics


In [ ]:
def per_protein(d):
    rows = []
    for p, g in d.groupby("protein"):
        rows.append((p, len(g), float(((g.label - g.prediction) ** 2).mean()), g.label.rank().corr(g.prediction.rank())))
    return pd.DataFrame(rows, columns=["protein", "n", "mse", "spearman"]).set_index("protein")

PP = {split: {arm: per_protein(d) for arm, d in AVG[split].items()} for split in SPLITS}
CLUSTER = {}
for split in SPLITS:
    with open(os.path.join(os.path.dirname(PATHS_BY_SPLIT[split].train_folds), "split_info.json")) as f:
        CLUSTER[split] = json.load(f)["cluster_of"]

print("Mean over held-out proteins (seed-averaged predictions; Spearman across drugs):")
rows = []
for split in SPLITS:
    for arm, t in PP[split].items():
        rows.append((split, arm, AVG[split][arm].attrs.get("n_seeds", "-"), t["mse"].mean(), t["spearman"].mean(), int(t["spearman"].isna().sum())))
summary = pd.DataFrame(rows, columns=["split", "arm", "seeds", "mean protein MSE", "mean protein Spearman", "proteins without Spearman"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)
print(summary.round(4).to_string(index=False))
os.makedirs(os.path.join(DRIVE_ROOT, "analysis"), exist_ok=True)
summary.to_csv(os.path.join(DRIVE_ROOT, "analysis", "per_protein_summary.csv"), index=False)

Mean over held-out proteins (seed-averaged predictions; Spearman across drugs):
       split                 arm seeds  mean protein MSE  mean protein Spearman  proteins without Spearman
cold_protein           drug_mean     -            0.6288                 0.4777                          0
cold_protein         random_1024     3            0.6661                 0.4733                          0
cold_protein           esmc_300m     3            0.4187                 0.6009                          0
cold_protein           esmc_600m     3            0.4347                 0.6005                          0
cold_protein prot_t5_xl_uniref50     3            0.5063                 0.5779                          0
cold_protein           structure     3            0.6525                 0.4977                          0
cold_protein              hybrid     3            0.5396                 0.5790                          0
cold_protein      hybrid_noedges     3            0.5000        

# Per cluster

In [ ]:
def cluster_table(split):
    held = sorted(PP[split]["drug_mean"].index)
    rows = []
    for cid in sorted({CLUSTER[split][p] for p in held}):
        members = [p for p in held if CLUSTER[split][p] == cid]
        row = {"cluster": cid, "proteins": len(members), "rows": int(PP[split]["drug_mean"].loc[members, "n"].sum()),
               "examples": ", ".join(members[:3]) + ("..." if len(members) > 3 else "")}
        for arm in PP[split]:
            row[f"{arm}|mse"] = PP[split][arm].loc[members, "mse"].mean()
            row[f"{arm}|rho"] = PP[split][arm].loc[members, "spearman"].mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values("rows", ascending=False)

short = {"drug_mean": "floor", "random_1024": "rand", "esmc_300m": "e300", "esmc_600m": "e600",
         "prot_t5_xl_uniref50": "t5", "structure": "struct", "hybrid": "hyb", "hybrid_noedges": "hybNE"}
for split in SPLITS:
    t = cluster_table(split)
    t.to_csv(os.path.join(DRIVE_ROOT, "analysis", f"per_cluster_{split}.csv"), index=False)
    show = t.rename(columns={c: short.get(c.split("|")[0], c.split("|")[0]) + "|" + c.split("|")[1] for c in t.columns if "|" in c})
    print(f"\n{split}: {len(t)} clusters" + (" (showing the 15 largest, all saved in the CSV)" if len(t) > 15 else ""))
    print(show.head(15).round(2).to_string(index=False))


cold_protein: 74 clusters (showing the 15 largest, all saved in the CSV)
 cluster  proteins  rows                                examples  floor|mse  floor|rho  rand|mse  rand|rho  e300|mse  e300|rho  e600|mse  e600|rho  t5|mse  t5|rho  struct|mse  struct|rho  hyb|mse  hyb|rho  hybNE|mse  hybNE|rho
      63        12   816 EGFR, EGFR(E746A750del), EGFR(G719C)...       1.72       0.27      1.80      0.25      1.17      0.62      1.16      0.66    1.65    0.53        1.90        0.26     1.80     0.51       1.46       0.65
     234         3   204           MET, MET(M1250T), MET(Y1235D)       0.92       0.57      0.97      0.52      0.72      0.66      0.72      0.63    0.78    0.65        1.03        0.47     0.87     0.63       1.03       0.55
     370         2   136                              PAK1, PAK3       0.27       0.55      0.27      0.58      0.17      0.59      0.19      0.61    0.21    0.61        0.36        0.49     0.29     0.57       0.17       0.61
      18         1

# Paired cluster bootstrap

In [ ]:
def boot_diff(split, a, b, metric, n_boot=5000, seed=0):
    d = PP[split][a][metric] - PP[split][b][metric]
    clusters = {}
    for p, v in d.items():
        if not np.isnan(v):
            clusters.setdefault(CLUSTER[split][p], []).append(v)
    groups = [np.array(v) for v in clusters.values()]
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(groups), len(groups))
        means[i] = np.concatenate([groups[j] for j in idx]).mean()
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), len(groups)

PAIRS = [("esmc_300m", "drug_mean"), ("esmc_600m", "drug_mean"), ("prot_t5_xl_uniref50", "drug_mean"),
         ("structure", "drug_mean"), ("random_1024", "drug_mean"),
         ("esmc_300m", "structure"), ("hybrid", "structure"), ("hybrid", "esmc_300m"),
         ("hybrid", "hybrid_noedges"), ("hybrid_noedges", "esmc_300m"), ("esmc_300m", "random_1024"),
         ("esmc_300m", "esmc_600m"), ("esmc_300m", "prot_t5_xl_uniref50")]
rows = []
for split in SPLITS:
    for a, b in PAIRS:
        if a not in PP[split] or b not in PP[split]:
            continue
        for metric in ["spearman", "mse"]:
            m, lo, hi, k = boot_diff(split, a, b, metric)
            rows.append((split, f"{a} - {b}", metric, m, lo, hi, k, "yes" if (lo > 0 or hi < 0) else "no"))
boot = pd.DataFrame(rows, columns=["split", "A - B", "metric", "mean diff", "95% low", "95% high", "clusters", "excludes 0"])
print(boot.round(4).to_string(index=False))
boot.to_csv(os.path.join(DRIVE_ROOT, "analysis", "bootstrap_differences.csv"), index=False)

       split                           A - B   metric  mean diff  95% low  95% high  clusters excludes 0
cold_protein           esmc_300m - drug_mean spearman     0.1232   0.0675    0.1855        74        yes
cold_protein           esmc_300m - drug_mean      mse    -0.2101  -0.3057   -0.1159        74        yes
cold_protein           esmc_600m - drug_mean spearman     0.1228   0.0611    0.1943        74        yes
cold_protein           esmc_600m - drug_mean      mse    -0.1940  -0.2989   -0.0831        74        yes
cold_protein prot_t5_xl_uniref50 - drug_mean spearman     0.1002   0.0532    0.1455        74        yes
cold_protein prot_t5_xl_uniref50 - drug_mean      mse    -0.1225  -0.1709   -0.0824        74        yes
cold_protein           structure - drug_mean spearman     0.0200  -0.0008    0.0438        74         no
cold_protein           structure - drug_mean      mse     0.0238  -0.0283    0.0697        74         no
cold_protein         random_1024 - drug_mean spearman  